# STRATA Demo: Download and Load a Dataset Snapshot

This notebook downloads one snapshot from the STRATA dataset directly into this environment and loads it for analysis.

**Dataset:** [SST-TG-P1F4R3200](https://doi.ccs.ornl.gov/dataset/5be73ed1-f138-504e-9851-4dff0f465a1d) — Direct numerical simulation of decaying stably-stratified turbulence (Taylor-Green vortex initialisation, Pr = 1, Fr = 4, Re = 3200)

| | |
|---|---|
| **Variables** | `u1`, `u2`, `u3` (velocity components) and `r` (perturbed density) |
| **Grid** | 512 × 512 × 256 |
| **File size** | ~255 MB per variable |
| **Format** | 32-bit float, little-endian binary |

**What you need:** A free [Globus account](https://www.globus.org/) — you can sign up with an existing Google or institutional account.

---
## Step 1: Install dependencies

In [ ]:
!pip install globus-sdk requests tqdm -q

---
## Step 2: Configuration

Choose which time step and variables to download. All other values are pre-filled.

In [ ]:
import os

# --- Pre-filled: STRATA app and dataset settings ---
CLIENT_ID       = "d47db6dc-0428-4076-9a6e-31927d7c7704"
COLLECTION_ID   = "57618e0a-2c99-45ff-9694-24141b92fa17"
HTTPS_BASE      = "https://g-e320e6.63720f.75bc.data.globus.org"
COLLECTION_PATH = "/gen101/world-shared/doi-data/OLCF/202504/10.13139_OLCF_2530508"

# --- User settings ---
TIMESTEP   = 1002                       # time step index to download
VARIABLES  = ["u1", "u2", "u3", "r"]   # u1/u2/u3 = velocity components, r = perturbed density
OUTPUT_DIR = "./strata_data"            # where to save files (change as needed)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Files will be saved to: {os.path.abspath(OUTPUT_DIR)}")

---
## Step 3: Authenticate with Globus

Click the login URL, sign in with your Globus account, and paste the authorisation code back here.

In [ ]:
import globus_sdk

HTTPS_SCOPE = f"https://auth.globus.org/scopes/{COLLECTION_ID}/https"

client = globus_sdk.NativeAppAuthClient(CLIENT_ID)
client.oauth2_start_flow(requested_scopes=HTTPS_SCOPE)

print("Open this URL in your browser to log in:")
print()
print(client.oauth2_get_authorize_url())
print()

auth_code  = input("Paste the authorisation code here: ").strip()
token_resp = client.oauth2_exchange_code_for_tokens(auth_code)

https_token = None
for rs, data in token_resp.by_resource_server.items():
    if rs != "auth.globus.org":
        https_token = data["access_token"]
        break

assert https_token, "Authentication failed: no HTTPS token received."
print("Authentication successful.")

---
## Step 4: Download

The dataset is split into subdirectories of 1000 time steps each (e.g. `02_1001to2000`). The cell below computes the correct subdirectory from the time step index and downloads each variable file with a progress indicator.

Each file is ~255 MB. Downloading all four variables (~1 GB total) takes a few minutes.

In [7]:
import requests
from tqdm.auto import tqdm

FIELD_SIZE = 256 * 1024 * 1024  # ~256 MB per field file

def timestep_subdir(t):
    idx   = (t - 1) // 1000 + 1
    start = (idx - 1) * 1000 + 1
    end   = idx * 1000
    return f"{idx:02d}_{start}to{end}"

def download_file(url, output_path, token):
    headers = {"Authorization": f"Bearer {token}"}
    resp    = requests.get(url, headers=headers, stream=True)
    resp.raise_for_status()

    total = int(resp.headers.get("content-length", FIELD_SIZE))
    with open(output_path, "wb") as f, tqdm(
        total=total,
        unit="B", unit_scale=True, unit_divisor=1024,
        desc=os.path.basename(output_path)
    ) as bar:
        for chunk in resp.iter_content(chunk_size=4 * 1024 * 1024):
            f.write(chunk)
            bar.update(len(chunk))


subdir           = timestep_subdir(TIMESTEP)
downloaded_files = {}

print(f"Time step {TIMESTEP} → subdirectory: {subdir}\n")

for var in VARIABLES:
    filename    = f"{var}.{TIMESTEP}"
    url         = f"{HTTPS_BASE}{COLLECTION_PATH}/{subdir}/{filename}?download=1"
    output_path = os.path.join(OUTPUT_DIR, filename)

    download_file(url, output_path, https_token)
    downloaded_files[var] = output_path

print("\nAll files downloaded.")

Time step 1002 → subdirectory: 02_1001to2000



u1.1002:   0%|          | 0.00/256M [00:00<?, ?B/s]

KeyboardInterrupt: 

---
## Step 5: Load the data

Each file contains a single field stored as 32-bit little-endian floats, with **x varying fastest** and **z slowest**. The x-dimension is padded to `Nx + 2 = 514` with two trailing zeros (stripped on load).

After loading, each array has shape `(Nz, Ny, Nx)` = `(256, 512, 512)`.

In [ ]:
import numpy as np

Nx, Ny, Nz = 512, 512, 256

def load_field(path):
    raw = np.fromfile(path, dtype="<f4")
    return raw.reshape(Nz, Ny, Nx + 2)[:, :, :Nx]

fields = {var: load_field(path) for var, path in downloaded_files.items()}

for var, arr in fields.items():
    print(f"  {var}:  shape {arr.shape},  min {arr.min():.4f},  max {arr.max():.4f}")

---
The fields are now loaded as NumPy arrays in the `fields` dictionary and ready for analysis.